<a href="https://colab.research.google.com/github/halakhalil5/shape-e-colab/blob/main/sample_to_3d_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shap-E

Objective: to generate 3D objects conditioned on text or images.

Reference: https://github.com/openai/shap-e

## Install Python packages

In [ ]:
!pip install shap-e torch torchvision numpy fastapi uvicorn pyngrok nest_asyncio pydantic

## Prepare the models

In [ ]:
import torch
# Shap-E imports (keep yours)
import numpy as np

from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

import os
import tempfile
import threading
from uuid import uuid4

from pyngrok import ngrok

from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import create_pan_cameras, decode_latent_images, gif_widget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

xm = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


## Define a wrapper

In [ ]:
def sample_text_to_3d(prompt):
    batch_size = 1
    guidance_scale = 15.0

    latents = diffusion.sample(
        model,
        batch_size=batch_size,
        model_kwargs=dict(texts=[prompt]),
        guidance_scale=guidance_scale,
        device=device,
    )

    return latents[0]

## Sample a 3D model

### Conditioned on a text prompt

In [ ]:

def save_model(latent, file_path):
    from shap_e.util.notebooks import decode_latent_mesh

    mesh = decode_latent_mesh(xm, latent).tri_mesh()

    with open(file_path, "wb") as f:
        mesh.write_glb(f)

Fast API


In [ ]:
@app.post("/generate_shap_e")
def generate_model(request: PromptRequest):
    try:
        prompt = request.prompt

        file_path = os.path.join(output_dir, f"{uuid4()}.glb")
        print("Saving to:", file_path)

        # ===== SHAP-E GENERATION =====
        latents = sample_latents(
            model=model,
            diffusion=diffusion,
            guidance_scale=15.0,
            batch_size=1,
            model_kwargs=dict(texts=[prompt]),
            progress=True,
        )

        for latent in latents:
            mesh = decode_latent_mesh(xm, latent).tri_mesh()

            with open(file_path, "wb") as f:
                mesh.write_glb(f)
        # =============================

        print("File exists:", os.path.exists(file_path))

        if not os.path.exists(file_path):
            return {"error": "File was not generated"}

        return FileResponse(file_path, media_type='model/gltf-binary')

    except Exception as e:
        return {"error": str(e)}

NGROK

In [ ]:
ngrok.set_auth_token("3BMDJ65WilSJSZ1xJYha3cUVPhw_4akNpGreTgEXsSwnXwjdD")

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

In [ ]:
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()

In [ ]:
import requests
url = public_url.public_url

r = requests.post(
    url + "/generate_shap_e",
    json={"prompt": "low poly wooden chair"}
)

print(r.status_code)
print(r.headers.get("content-type"))